In [3]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    GridSearchCV, StratifiedKFold, cross_validate
)
from sklearn.metrics import make_scorer, recall_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

## Section 1 — Load prepared datasets

Two versions are loaded:
- `with_leakage`: keeps post-diagnosis and same-pregnancy features for **benchmarking only**.
- `no_leakage`: only features available before the risk decision is made — this is the **deployable** version.

In [4]:
def load_version(prefix):
    train = pd.read_csv(f"../data/pregnancy_train_{prefix}_features.csv")
    test  = pd.read_csv(f"../data/pregnancy_test_{prefix}_features.csv")

    y_train = train.pop("Risk Level").astype(int)
    y_test  = test.pop("Risk Level").astype(int)

    numeric_cols = train.select_dtypes(include="number").columns.tolist()
    X_train = train[numeric_cols].fillna(train[numeric_cols].median())
    X_test  = test[numeric_cols].fillna(train[numeric_cols].median())
    return X_train, X_test, y_train, y_test

versions = {}
for version in ["with_leakage", "no_leakage"]:
    X_train, X_test, y_train, y_test = load_version(version)
    versions[version] = dict(X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test)
    print(f"{version:>13}  | train={X_train.shape}  test={X_test.shape}  | pos_rate_train={y_train.mean():.3f}")

 with_leakage  | train=(934, 12)  test=(237, 12)  | pos_rate_train=0.405
   no_leakage  | train=(934, 10)  test=(237, 10)  | pos_rate_train=0.405


## Section 2 — Cross-validation setup and model definitions

- `StratifiedKFold(n_splits=5)` preserves the class ratio in every fold.
- `recall_high_scorer` tunes and selects on High-risk Recall.
- `scoring` reports Accuracy, Recall and ROC-AUC for every model, so we can see the trade-offs.

In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
recall_high_scorer = make_scorer(recall_score, pos_label=1)
scoring = {
    "accuracy": "accuracy",
    "recall_high": recall_high_scorer,
    "roc_auc": "roc_auc",
}

models = {
    "LogisticRegression": LogisticRegression(
        class_weight="balanced", max_iter=2000, random_state=42
    ),
    "DecisionTree": DecisionTreeClassifier(
        class_weight="balanced", random_state=42
    ),
    "RandomForest": RandomForestClassifier(
        class_weight="balanced", random_state=42, n_jobs=-1
    ),
    "GradientBoosting": GradientBoostingClassifier(random_state=42),
}

param_grids = {
    "LogisticRegression": {"C": [0.01, 0.1, 1, 10]},
    "DecisionTree":       {"max_depth": [3, 5, 7, None]},
    "RandomForest":       {"n_estimators": [100, 300], "max_depth": [5, 10, None]},
    "GradientBoosting":   {"n_estimators": [100, 200], "max_depth": [2, 3]},
}

## Section 3 — Comparison helper

For each model:
1. Run `GridSearchCV` to find the best hyperparameters on Recall.
2. Re-run `cross_validate` on the tuned estimator to get mean and std across folds for all three metrics.

The result table is sorted by mean Recall (descending).

In [6]:
def compare_models(X_train, y_train, cv=cv):
    rows, best_models, best_params = [], {}, {}
    for name, estimator in models.items():
        search = GridSearchCV(
            estimator,
            param_grids[name],
            scoring=recall_high_scorer,
            cv=cv,
            n_jobs=-1,
            refit=True,
        )
        search.fit(X_train, y_train)
        best_models[name] = search.best_estimator_
        best_params[name] = search.best_params_

        scores = cross_validate(
            search.best_estimator_, X_train, y_train,
            cv=cv, scoring=scoring, n_jobs=-1,
        )

        rows.append({
            "Model": name,
            "Accuracy mean": scores["test_accuracy"].mean(),
            "Accuracy std":  scores["test_accuracy"].std(),
            "Recall mean":   scores["test_recall_high"].mean(),
            "Recall std":    scores["test_recall_high"].std(),
            "ROC-AUC mean":  scores["test_roc_auc"].mean(),
            "ROC-AUC std":   scores["test_roc_auc"].std(),
        })

    return (
        pd.DataFrame(rows).sort_values("Recall mean", ascending=False).reset_index(drop=True),
        best_models,
        best_params,
    )

## Section 4 — Run the full comparison for both versions

The best model per version is selected as the one with the highest mean High-risk Recall across folds, then saved along with its test set for later evaluation.

In [7]:
results_by_version = {}
models_by_version  = {}
params_by_version  = {}
best_name_by_version = {}

for version, data in versions.items():
    X_train, X_test = data["X_train"], data["X_test"]
    y_train, y_test = data["y_train"], data["y_test"]

    print("=" * 70)
    print(f"VERSION: {version}")
    print("=" * 70)

    table, best_models, best_params = compare_models(X_train, y_train)
    results_by_version[version]  = table
    models_by_version[version]   = best_models
    params_by_version[version]   = best_params

    display(table.round(4))
    print("Best hyperparameters:")
    for k, v in best_params.items():
        print(f"  {k:>20}: {v}")

    best_name = table.iloc[0]["Model"]
    best_name_by_version[version] = best_name
    joblib.dump(best_models[best_name], f"best_model_{version}.pkl")
    joblib.dump(X_test,                f"X_test_{version}.pkl")
    joblib.dump(y_test,                f"y_test_{version}.pkl")
    joblib.dump(best_name,             f"best_model_name_{version}.pkl")
    print(f"Selected best model: {best_name}")
    print()

VERSION: with_leakage


,Model,Accuracy mean,Accuracy std,Recall mean,Recall std,ROC-AUC mean,ROC-AUC std
0,GradientBoosting,0.9914,0.0073,0.9947,0.0064,0.9996,0.0003
1,RandomForest,0.9839,0.0123,0.9895,0.0098,0.9994,0.0004
2,LogisticRegression,0.9647,0.0073,0.9657,0.0243,0.9951,0.0025
3,DecisionTree,0.9604,0.0080,0.9631,0.0210,0.9723,0.0126


Best hyperparameters:
    LogisticRegression: {'C': 10}
          DecisionTree: {'max_depth': 5}
          RandomForest: {'max_depth': 10, 'n_estimators': 100}
      GradientBoosting: {'max_depth': 3, 'n_estimators': 100}
Selected best model: GradientBoosting

VERSION: no_leakage


,Model,Accuracy mean,Accuracy std,Recall mean,Recall std,ROC-AUC mean,ROC-AUC std
0,GradientBoosting,0.9861,0.0043,0.9815,0.0065,0.9986,0.0010
1,RandomForest,0.9786,0.0084,0.9735,0.0002,0.9982,0.0012
2,DecisionTree,0.9615,0.0114,0.9603,0.0188,0.9641,0.0097
3,LogisticRegression,0.9454,0.0104,0.9392,0.0134,0.9868,0.0043


Best hyperparameters:
    LogisticRegression: {'C': 10}
          DecisionTree: {'max_depth': 7}
          RandomForest: {'max_depth': 10, 'n_estimators': 300}
      GradientBoosting: {'max_depth': 3, 'n_estimators': 100}
Selected best model: GradientBoosting



### Question 1 — Compare the four algorithms

The two tables above report mean and standard deviation for **Accuracy**, **High-risk Recall**, and **ROC-AUC** for Logistic Regression, Decision Tree, Random Forest, and Gradient Boosting, evaluated under an identical 5-fold Stratified CV setup for both dataset versions.

### Question 2 — Best hyperparameters and why Recall was used

The `params_by_version` dictionary reports the selected settings for every model.

**High-risk Recall** was used as the tuning and selection criterion because missing a truly high-risk pregnancy is far more consequential than generating an avoidable referral. Accuracy alone would be misleading given the moderate class imbalance, and ROC-AUC is threshold-independent, whereas the clinical decision is threshold-bound.

### Question 3 — Standard deviation of Recall (stability)

The `Recall std` column quantifies fold-to-fold variability. Lower std means the model's recall is not sensitive to which particular fold it is evaluated on — i.e. more stable performance than a single split can demonstrate.

In [8]:
stability_rows = []
for version, table in results_by_version.items():
    best = table.iloc[0]
    stability_rows.append({
        "Version":       version,
        "Best model":    best["Model"],
        "Recall mean":   round(best["Recall mean"], 4),
        "Recall std":    round(best["Recall std"], 4),
        "Accuracy mean": round(best["Accuracy mean"], 4),
        "ROC-AUC mean":  round(best["ROC-AUC mean"], 4),
    })

stability_df = pd.DataFrame(stability_rows)
stability_df

,Version,Best model,Recall mean,Recall std,Accuracy mean,ROC-AUC mean
0,with_leakage,GradientBoosting,0.9947,0.0064,0.9914,0.9996
1,no_leakage,GradientBoosting,0.9815,0.0065,0.9861,0.9986


### Question 4 — Impact of data leakage

The same comparison is repeated on the leakage-inclusive and leakage-excluded datasets. The difference quantifies how much of the "apparent" performance comes from features that would not be available in a real deployment.

In [9]:
comparison = results_by_version["with_leakage"].merge(
    results_by_version["no_leakage"],
    on="Model",
    suffixes=(" with_leakage", " no_leakage"),
)

impact_df = pd.DataFrame({
    "Model": comparison["Model"],
    "Recall mean (with)":    comparison["Recall mean with_leakage"],
    "Recall mean (no)":      comparison["Recall mean no_leakage"],
    "Recall delta":          comparison["Recall mean with_leakage"] - comparison["Recall mean no_leakage"],
    "Accuracy delta":        comparison["Accuracy mean with_leakage"] - comparison["Accuracy mean no_leakage"],
    "ROC-AUC delta":         comparison["ROC-AUC mean with_leakage"] - comparison["ROC-AUC mean no_leakage"],
})

impact_df.round(4)

,Model,Recall mean (with),Recall mean (no),Recall delta,Accuracy delta,ROC-AUC delta
0,GradientBoosting,0.9947,0.9815,0.0133,0.0053,0.0010
1,RandomForest,0.9895,0.9735,0.0159,0.0053,0.0012
2,LogisticRegression,0.9657,0.9392,0.0266,0.0193,0.0083
3,DecisionTree,0.9631,0.9603,0.0028,-0.0011,0.0082


**Interpretation:**

- Positive deltas mean the leakage-inclusive version looked better than it truly is.
- The `no_leakage` numbers are the honest, deployable estimate of performance.
- If the delta is large, it confirms that the excluded features were carrying information about the target that would not be available at prediction time.

## Section 5 — Optional: final test-set evaluation of the selected model

This is the single use of the held-out test set. It was never seen during tuning or cross-validation.

In [10]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

final_rows = []
for version in ["with_leakage", "no_leakage"]:
    model = joblib.load(f"best_model_{version}.pkl")
    X_test = joblib.load(f"X_test_{version}.pkl")
    y_test = joblib.load(f"y_test_{version}.pkl")

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    cm = confusion_matrix(y_test, y_pred)
    final_rows.append({
        "Version":        version,
        "Best model":     joblib.load(f"best_model_name_{version}.pkl"),
        "Accuracy":       round((y_pred == y_test).mean(), 4),
        "Recall (High)":  round(recall_score(y_test, y_pred, pos_label=1), 4),
        "ROC-AUC":        round(roc_auc_score(y_test, y_prob), 4),
        "TN": cm[0, 0], "FP": cm[0, 1],
        "FN": cm[1, 0], "TP": cm[1, 1],
    })

    print("=" * 60)
    print(f"{version} — test-set classification report")
    print("=" * 60)
    print(classification_report(y_test, y_pred, digits=4))

pd.DataFrame(final_rows)

with_leakage — test-set classification report
              precision    recall  f1-score   support

           0     0.9860    0.9930    0.9895       142
           1     0.9894    0.9789    0.9841        95

    accuracy                         0.9873       237
   macro avg     0.9877    0.9860    0.9868       237
weighted avg     0.9874    0.9873    0.9873       237

no_leakage — test-set classification report
              precision    recall  f1-score   support

           0     0.9658    0.9930    0.9792       142
           1     0.9890    0.9474    0.9677        95

    accuracy                         0.9747       237
   macro avg     0.9774    0.9702    0.9735       237
weighted avg     0.9751    0.9747    0.9746       237



,Version,Best model,Accuracy,Recall (High),ROC-AUC,TN,FP,FN,TP
0,with_leakage,GradientBoosting,0.9873,0.9789,0.9993,141,1,2,93
1,no_leakage,GradientBoosting,0.9747,0.9474,0.9989,141,1,5,90


## Final Summary

- **Model selection criterion:** highest mean High-risk Recall over 5-fold Stratified CV.
- **Why Recall:** missing a high-risk pregnancy is more costly than a false positive referral.
- **Leakage:** models trained with leakage-prone features consistently overstate performance; the `no_leakage` numbers are the honest, deployable estimate.
- **Stability:** `Recall std` across folds reports how sensitive each model is to the specific fold split.
- **Saved artifacts per version:**
  - `best_model_{version}.pkl`
  - `best_model_name_{version}.pkl`
  - `X_test_{version}.pkl`
  - `y_test_{version}.pkl`